# Exercise 04 - The `SourceConnector` contract: point-and-pull without a pipeline per source

**Project Aegis - Phase 1 (Data & Storage) - learning loop step 2 (capstone)**

### Why this exercise
You now have three moving parts: pull a claim (01), generate a ledger (02), store & query it (03).
Left as-is, every new data source would mean new bespoke glue - and that doesn't scale as sources
multiply (banks -> other sectors -> contracts -> call reports). This exercise solves that with the
design from **ADR-0001**: one **descriptor** (a plain dict saying *what* to pull) and one **connector**
per source *type* that knows *how*. A single `ingest(descriptor)` entry point dispatches to the right
connector. **Adding a source becomes adding an adapter, not a pipeline** (spec FR-IN-4 / FR-IN-5).

### What you'll learn
- **The descriptor** - declarative, reproducible, the provenance root of a pull (FR-IN-5).
- **The connector contract** - `resolve -> fetch -> normalize -> emit` behind one `pull()` method.
- **A registry + dispatcher** - the pattern that turns N sources into one uniform call site.
- **A canonical envelope** - so downstream code never branches on where data came from.
- The payoff: wire 01 and 02 behind the contract, run SG-1 through it, then add a *new* source in ~6 lines.

### Requirements
`pip install faker`; internet optional (the SEC connector has an offline fallback).

## Setup - the building blocks from 01 & 02 (given)

These are the Exercise 01 claim-pull and Exercise 02 ledger generator, unchanged - we're about to
wrap them, not rewrite them.

In [ ]:
import urllib.request, json, random
from decimal import Decimal
from dataclasses import dataclass, field
from faker import Faker

SEC_USER_AGENT = {"User-Agent": "Aegis Learning Project sunitsingh.bitsg@gmail.com"}

def _sec_pull(ticker, concept):
    def get(u):
        return json.load(urllib.request.urlopen(urllib.request.Request(u, headers=SEC_USER_AGENT), timeout=60))
    tbl = get("https://www.sec.gov/files/company_tickers.json")
    cik = next(str(r["cik_str"]).zfill(10) for r in tbl.values() if r["ticker"].upper() == ticker.upper())
    facts = get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json")
    rec = max((e for e in facts["facts"]["us-gaap"][concept]["units"]["USD"] if e.get("form") == "10-K"),
              key=lambda e: e["end"])
    return facts["entityName"], cik, rec

def make_ledger(target_total, n, seed, account):
    rng = random.Random(seed); fake = Faker(); fake.seed_instance(seed)
    tc = int((target_total * 100).to_integral_value())
    w = [rng.random() for _ in range(n)]; s = sum(w)
    cents = [int(tc * x / s) for x in w]; cents[-1] += tc - sum(cents)
    return [{"txn_id": f"{account[:3].upper()}-{seed}-{i:05d}", "date": fake.date_between(start_date='-1y').isoformat(),
             "counterparty": fake.company(), "account": account, "amount": Decimal(cents[i]) / 100} for i in range(n)]

print("building blocks ready")

## The canonical envelope + the contract (given)

Every connector, whatever its source, returns the **same** container - an `IngestionResult`. That
uniformity is the point: downstream code (storage, reconciliation, the UI) reads `result.records`
and `result.kind` without ever knowing or caring whether it came from XBRL, a simulated ledger, or a
bank call report.

The base `SourceConnector` declares one method, `pull(descriptor) -> IngestionResult`. Each concrete
connector sets a `source` string and implements `pull` as `resolve -> fetch -> normalize`.

In [ ]:
@dataclass
class IngestionResult:
    source: str          # source type, e.g. 'sec-xbrl'
    kind: str            # 'claim' or 'ledger' - what downstream should do with it
    entity: str          # whose data this is
    records: list        # the normalized payload (a claim dict, or ledger rows)
    descriptor: dict     # the request that produced this - provenance root (FR-IN-5)
    provenance: dict = field(default_factory=dict)

class SourceConnector:
    source = None        # each subclass sets its source-type string
    def pull(self, descriptor: dict) -> IngestionResult:
        raise NotImplementedError

# --- the registry: source-type string -> a connector instance ---
CONNECTORS = {}
def register(cls):
    CONNECTORS[cls.source] = cls()
    return cls

print("contract defined")

## Step 1 - the dispatcher (gap 1)

`ingest` is the single front door: hand it any descriptor and it routes to the right connector by the
descriptor's `source` field. This is the line that makes *all* sources share one call site - the
scaling win. (Compare: without it, every caller would `if source == ...` branch by hand.)

**Your task (gap 1):** look up the connector for `descriptor['source']` in `CONNECTORS` and return
its `.pull(descriptor)`. Raise a clear error if no connector is registered for that source.

In [ ]:
def ingest(descriptor: dict) -> IngestionResult:
    """The single ingestion entry point: dispatch a descriptor to its connector."""
    # ==== YOUR CODE HERE (start) ====
    # 1) read source = descriptor['source']
    # 2) find CONNECTORS[source]; if missing, raise ValueError listing known sources
    # 3) return connector.pull(descriptor)
    raise NotImplementedError
    # ==== YOUR CODE HERE (end) ====

print("dispatcher defined")

## Step 2 - wrap Exercise 01 as the `sec-xbrl` connector (given)

Here the `resolve -> fetch -> normalize` shape is explicit. Note it emits the canonical envelope with
`kind='claim'` and carries the filing's accession as provenance - exactly the audit-trail grounding.

In [ ]:
@register
class SecXbrlConnector(SourceConnector):
    source = "sec-xbrl"
    def pull(self, d: dict) -> IngestionResult:
        try:
            entity, cik, rec = _sec_pull(d["ticker"], d["concept"])           # resolve + fetch
        except Exception as e:
            print("  (offline, recorded JPM claim)", e)
            entity, cik, rec = "JPMORGAN CHASE & CO", "0000019617", {
                "val": 4062462000000, "end": "2025-12-31", "fy": 2025,
                "form": "10-K", "accn": "0001628280-26-008131"}
        claim = {                                                              # normalize
            "concept": d["concept"], "claimed_value": rec["val"], "period_end": rec["end"],
            "provenance": {"form": rec["form"], "fy": rec.get("fy"), "accession": rec["accn"]}}
        return IngestionResult("sec-xbrl", "claim", entity, [claim], d, claim["provenance"])

print("registered:", list(CONNECTORS))

## Step 3 - wrap Exercise 02 as the `simulated-ledger` connector (gap 2)

Now you build the other adapter. The generation is done (`make_ledger`); your job is to return the
**canonical envelope** so this source looks identical to every other one downstream. Storing the
`seed` in provenance is what makes the pull reproducible (FR-IN-5).

**Your task (gap 2):** return an `IngestionResult` with `source='simulated-ledger'`, `kind='ledger'`,
the descriptor's `entity`, the generated `rows` as records, the descriptor `d`, and a provenance dict
recording the `seed` and row count.

In [ ]:
@register
class SimulatedLedgerConnector(SourceConnector):
    source = "simulated-ledger"
    def pull(self, d: dict) -> IngestionResult:
        rows = make_ledger(Decimal(str(d["target_total"])), d["n"], d["seed"], d["account"])
        # ==== YOUR CODE HERE (start) ====
        # return IngestionResult(source='simulated-ledger', kind='ledger', entity=d['entity'],
        #                        records=rows, descriptor=d,
        #                        provenance={'seed': d['seed'], 'rows': len(rows)})
        raise NotImplementedError
        # ==== YOUR CODE HERE (end) ====

# --- self-check ---
r = ingest({"source": "simulated-ledger", "entity": "ACME", "target_total": "1000000.00",
            "n": 10, "seed": 1, "account": "operating_lease_liability"})
assert isinstance(r, IngestionResult) and r.kind == "ledger" and len(r.records) == 10
assert r.provenance["seed"] == 1
print("PASS - simulated-ledger connector returns the canonical envelope")

## Step 4 - SG-1 through the contract

Now the whole MVP flows through one uniform interface. We `ingest` a real claim and an anchored
(discrepant) ledger with the **same call**, then reconcile - the data's origin is invisible to the
audit logic.

In [ ]:
claim_res = ingest({"source": "sec-xbrl", "ticker": "JPM", "concept": "Liabilities"})
claimed = Decimal(claim_res.records[0]["claimed_value"])

ledger_res = ingest({"source": "simulated-ledger", "entity": claim_res.entity,
                     "target_total": str(claimed - Decimal("125000000.00")),
                     "n": 500, "seed": 7, "account": "operating_lease_liability"})

def reconcile(claim_res, ledger_res):
    claimed = Decimal(claim_res.records[0]["claimed_value"])
    total = sum((r["amount"] for r in ledger_res.records), Decimal("0"))
    delta = claimed - total
    return {"entity": claim_res.entity, "status": "MATCH" if delta == 0 else "DISCREPANCY",
            "claimed": claimed, "ledger_total": total, "delta": delta,
            "claim_provenance": claim_res.provenance, "ledger_provenance": ledger_res.provenance}

f = reconcile(claim_res, ledger_res)
print(f"[{f['status']}] {f['entity']}: claim ${f['claimed']:,} vs ledger ${f['ledger_total']:,}")
print(f"  delta ${f['delta']:,}  | claim accn {f['claim_provenance']['accession']}  | ledger seed {f['ledger_provenance']['seed']}")

## Step 5 - the payoff: add a brand-new source in ~6 lines (given)

This is the scaling claim, made literal. A bank's regulatory **Call Report** (filed to the FFIEC) is
a different source entirely - but onboarding it is just a new adapter + `@register`. **`ingest` does
not change. No caller changes.** That's FR-IN-4.

In [ ]:
@register
class FfiecCallReportConnector(SourceConnector):
    source = "ffiec-callreport"
    def pull(self, d: dict) -> IngestionResult:
        # (stub) a real version would hit the FFIEC API by RSSD id; shape matches every other source
        claim = {"concept": d["concept"], "claimed_value": d["value"],
                 "provenance": {"source": "FFIEC", "rssd": d["rssd"]}}
        return IngestionResult("ffiec-callreport", "claim", d["entity"], [claim], d, claim["provenance"])

print("known sources now:", list(CONNECTORS))
cr = ingest({"source": "ffiec-callreport", "entity": "SOME BANK", "rssd": 480228,
             "concept": "TotalDeposits", "value": 123456000000})
print("ingested via the same front door:", cr.kind, cr.entity, f"${Decimal(cr.records[0]['claimed_value']):,}")

## Step 6 - reproducibility (FR-IN-5)

Because a descriptor fully determines a pull (and the ledger is seeded), re-running the same
descriptor yields identical data - the property that lets a finding be re-derived months later.

In [ ]:
d = {"source": "simulated-ledger", "entity": "ACME", "target_total": "500000.00",
     "n": 100, "seed": 2025, "account": "operating_lease_liability"}
a = ingest(d); b = ingest(d)
assert [r["amount"] for r in a.records] == [r["amount"] for r in b.records]
assert [r["counterparty"] for r in a.records] == [r["counterparty"] for r in b.records]
print("PASS - same descriptor -> identical ledger (reproducible)")

## Recap - Phase 1 exercises complete

You unified the whole data layer behind one contract:
- a **descriptor** describes *what* to pull (declarative, reproducible),
- a **connector** per source knows *how* (`resolve -> fetch -> normalize`),
- a **registry + `ingest`** give every source one uniform call site,
- a **canonical envelope** lets the audit logic ignore data origin,
- and SG-1 now runs end-to-end through the contract, with a new source addable in ~6 lines.

This is the architecture that will become `src/` ingestion. Next in the loop: the **Phase 1
checkpoint quiz**, then we **commit** the whole phase and start building the real package - swapping
SQLite for Postgres and these notebook functions for tested modules.

**Stretch goals (optional):**
1. Add a `to_sql_rows()` helper on a `kind=='ledger'` result that yields the tuples for Exercise 03's
   parameterized INSERT - connecting this contract to the database layer.
2. Make `register` reject two connectors claiming the same `source` (fail fast on a config error).
3. Give `IngestionResult` a `pulled_at` field - but pass the timestamp *in* via the descriptor, so
   pulls stay reproducible (a subtle but real determinism point).